# B-16 round 2: TICA embeddings + static AR-lag features for TabICLv2 (Track B)

**Purpose.** Revisits forecasting per user request, focused on **Track B** (TabICLv2, zero-shot)
rather than Track A's trained trees. Tests two new additive feature families on top of the
standing champion config (`BASE+species`, F-10/D-67, MASE=0.715 climatology-scored, D-80):

1. **`+static_ar`** -- pre-anchor summary statistics of the CH4 target itself (mean/std/max/trend/
   autocorrelation/days-since-spike), computed once per anchor and held constant across the whole
   365-day rollout. TabPFN/TabICLv2 already see the raw `y_observed` history natively as context
   (unlike Track A's trees, which need lags spelled out) -- this tests whether explicit derived
   summary signal helps anyway. Chosen over two more expensive alternatives after weighing cost:
   a genuine day-by-day recursive rollout (~365x more API calls, ~30h+ for a full sweep) and a
   block-recursive version (~13x more calls, ~75-90min) were both considered; static features were
   chosen as the first, cheapest test, with the explicit acknowledged tradeoff that they go stale
   the further into the 365-day window a prediction lands (a December anchor's "recent regime"
   features are still describing December when predicting the following June).
2. **`+tica`** -- time-lagged independent component analysis (TICA) embeddings of the continuous
   driver set, reusing the exact method D-79 (`temp_gap_filing_exploration copy.ipynb`, section 15)
   built for gap-filling: solves the generalized eigenvalue problem `C_tau @ v = lambda * C_0 @ v`
   on symmetrized lag-`tau` covariance matrices, from scratch (no `deeptime` dependency, matching
   that precedent). **Known prior result to set expectations**: D-79 found TICA-as-feature was a
   "wash" for gap-filling (RF/TabICL) -- this is a different task (extrapolation, not interpolation)
   so not assumed to transfer, but worth remembering going in.

**Adapted for daily (not hourly) data**: `tau=7` (1 week, vs. D-79's `tau=24` hours/1 day) to look
past day-to-day noise toward slower seasonal/management dynamics. Same correctness precautions
D-79 established: z-score features first (TICA is scale-sensitive), use temporally contiguous
daily rows (not sparse target-valid-only rows), and exclude deterministic calendar encodings
(`fx_DOY_sin/cos`) that would otherwise risk a trivial tau-alignment artifact (D-79 hit exactly
this bug with an hour-of-day sine at `tau=24`).

**No-leakage discipline**: both new families are fit **per anchor, on pre-anchor history only**
(same convention as this project's AOA/conformal-calibration fitting everywhere else), then applied
to that anchor's own future window.

**A real bug caught by the smoke test, not silently worked around**: the first version broadcast
the static-AR scalar as a constant column across the ENTIRE historical context, not just the
future window -- this gave TabICL zero-variance history columns, which broke an internal feature-
deduplication filter (`IndexError: boolean index did not match indexed array`) on every
`+static_ar` config. Fixed by giving the historical context genuinely time-varying rolling
statistics (`historical_static_ar_frame()`, vectorized, each day using only data strictly before
it) -- only the future window stays frozen-at-anchor, which was always the intended design.

**Scope**: TabICLv2 only (user-selected, Track B), all 3 towers x 5 anchors (2018-2022) -- full
coverage per this project's standing convention. Baseline + 3 new configs (`+static_ar`, `+tica`,
`+static_ar+tica`) = 4 configs x 3 towers x 5 anchors = 60 calls total.

**Scoring**: `rr.bin_metrics()`, per lead-time bin, per config/tower/anchor. Climatology-scored MASE
is primary (D-80 standing convention); R2/persistence-MASE also reported for continuity with
existing tables.


## 1. Setup

In [1]:
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
from scipy.linalg import eigh as gen_eigh

warnings.filterwarnings("ignore")

ROOT = r"c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + r"\src")

import models.recursive_rollout as rr

HOURLY = rf"{ROOT}\data\Hourly"
RESULTS = rf"{ROOT}\results"

TOWERS = [2, 4, 9]
N_DAYS = 365
ANCHOR_YEARS = [2018, 2019, 2020, 2021, 2022]

dv = pd.read_csv(f"{HOURLY}/forecast_daily_v3.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in TOWERS}
print("Loaded forecast_daily_v3.csv:", dv.shape, "towers:", {t: len(T[t]) for t in TOWERS})


Loaded forecast_daily_v3.csv: (8772, 66) towers: {2: 2924, 4: 2924, 9: 2924}


## 2. Baseline feature config (BASE+species, the standing champion)

Reproduces `b16_foundation_models_v3.py`'s exact `FAMILIES`/`BASE_FX` construction (imported in
spirit, not retyped independently, so this can't silently drift from what "species"/"BASE" mean
there) -- this is the control every new family is tested against.

In [2]:
SPECIES_COLS = ["fx_cattle_dens", "fx_sheep_dens", "fx_lamb_dens"]

fx_all = [c for c in dv.columns if c.startswith("fx")]
BASE_FX = [c for c in fx_all if c not in SPECIES_COLS]
CHAMPION_FX = BASE_FX + SPECIES_COLS
print(f"BASE+species: {len(CHAMPION_FX)} columns ({len(BASE_FX)} base + {len(SPECIES_COLS)} species)")


BASE+species: 52 columns (49 base + 3 species)


## 3. Static AR-lag features (`+static_ar`)

10 scalar features computed **once per (tower, anchor) from pre-anchor `y_observed` only**, then
broadcast as a constant value across the entire 365-day future window. Explicit, stated tradeoff:
these go stale the further into the rollout a prediction lands (see notebook intro).

In [3]:
def static_ar_features(hist_target, anchor, spike_q=0.9):
    """hist_target: real y_observed Series, index up to (not including) anchor.
    Returns a dict of 10 scalar features describing the pre-anchor regime."""
    s = hist_target.dropna()
    out = {}
    for w in (7, 14, 28):
        window = s.loc[anchor - pd.Timedelta(days=w):anchor]
        out[f"fx_ch4_pre_mean{w}"] = float(window.mean()) if len(window) else np.nan
        out[f"fx_ch4_pre_std{w}"] = float(window.std()) if len(window) > 1 else np.nan

    w28 = s.loc[anchor - pd.Timedelta(days=28):anchor]
    out["fx_ch4_pre_max28"] = float(w28.max()) if len(w28) else np.nan

    spike_thresh = s.quantile(spike_q)
    recent_spikes = s.loc[(s > spike_thresh) & (s.index <= anchor)]
    out["fx_ch4_days_since_spike"] = float((anchor - recent_spikes.index.max()).days) if len(recent_spikes) else 365.0

    w14 = s.loc[anchor - pd.Timedelta(days=14):anchor]
    if len(w14) >= 2:
        x = np.arange(len(w14))
        out["fx_ch4_pre_trend14"] = float(np.polyfit(x, w14.values, 1)[0])
    else:
        out["fx_ch4_pre_trend14"] = 0.0

    w_ac = s.loc[anchor - pd.Timedelta(days=21):anchor]
    out["fx_ch4_pre_autocorr7"] = float(w_ac.autocorr(lag=7)) if len(w_ac) > 7 else 0.0

    return out


STATIC_AR_COLS = ["fx_ch4_pre_mean7", "fx_ch4_pre_mean14", "fx_ch4_pre_mean28",
                   "fx_ch4_pre_std7", "fx_ch4_pre_std14", "fx_ch4_pre_std28",
                   "fx_ch4_pre_max28", "fx_ch4_days_since_spike", "fx_ch4_pre_trend14",
                   "fx_ch4_pre_autocorr7"]


def historical_static_ar_frame(y_observed_series, spike_q=0.9):
    """Vectorized, genuinely time-varying version of static_ar_features() for the HISTORICAL
    context window -- one row per historical day, each using only data strictly before that day
    (via .shift(1)). This is deliberately NOT constant across history (only the future window is
    frozen-at-anchor, by design) -- an earlier version broadcast the same anchor-time scalar across
    the entire historical context too, which gave TabICL zero-variance columns and broke its
    internal feature-deduplication filter (caught by the smoke test's IndexError before the full
    sweep, not silently worked around)."""
    s = y_observed_series
    shifted = s.shift(1)
    df = pd.DataFrame(index=s.index)
    for w in (7, 14, 28):
        df[f"fx_ch4_pre_mean{w}"] = shifted.rolling(f"{w}D", min_periods=1).mean()
        df[f"fx_ch4_pre_std{w}"] = shifted.rolling(f"{w}D", min_periods=2).std()
    df["fx_ch4_pre_max28"] = shifted.rolling("28D", min_periods=1).max()

    spike_thresh = s.quantile(spike_q)
    spike_date = pd.Series(s.index, index=s.index).where(shifted > spike_thresh).ffill()
    days_since = (pd.Series(s.index, index=s.index) - spike_date).dt.days
    df["fx_ch4_days_since_spike"] = days_since.fillna(365.0)

    def _trend(w):
        w = w.dropna()
        return np.polyfit(np.arange(len(w)), w.values, 1)[0] if len(w) >= 2 else np.nan
    df["fx_ch4_pre_trend14"] = shifted.rolling("14D", min_periods=2).apply(_trend, raw=False)

    def _autocorr7(w):
        return w.autocorr(lag=7) if w.notna().sum() > 7 else np.nan
    df["fx_ch4_pre_autocorr7"] = shifted.rolling("21D", min_periods=8).apply(_autocorr7, raw=False)

    return df.fillna(0.0)  # early-history rows with insufficient window -> neutral 0, same as TICA's NaN handling


# Sanity check: Tower 4, anchor 2021 -- both the frozen future-window features and the
# time-varying historical frame
_anchor_test = pd.Timestamp("2021-12-16")
_feats_test = static_ar_features(T[4].loc[:_anchor_test, "y_observed"], _anchor_test)
print(f"{len(STATIC_AR_COLS)} static AR features, sanity check (T4, anchor 2021, frozen future value):")
for k, v in _feats_test.items():
    print(f"  {k}: {v:.3f}")

_hist_ar_test = historical_static_ar_frame(T[4].loc[:_anchor_test, "y_observed"])
print(f"\nHistorical (time-varying) frame shape: {_hist_ar_test.shape}, "
      f"unique fx_ch4_pre_mean7 values: {_hist_ar_test['fx_ch4_pre_mean7'].nunique()} "
      f"(should be >1, confirming NOT constant)")


10 static AR features, sanity check (T4, anchor 2021, frozen future value):
  fx_ch4_pre_mean7: 35.057
  fx_ch4_pre_std7: 9.536
  fx_ch4_pre_mean14: 22.049
  fx_ch4_pre_std14: 12.183
  fx_ch4_pre_mean28: 15.063
  fx_ch4_pre_std28: 11.994
  fx_ch4_pre_max28: 46.011
  fx_ch4_days_since_spike: 83.000
  fx_ch4_pre_trend14: 3.606
  fx_ch4_pre_autocorr7: -0.659



Historical (time-varying) frame shape: (1811, 10), unique fx_ch4_pre_mean7 values: 1247 (should be >1, confirming NOT constant)


## 4. TICA embeddings (`+tica`)

Reuses D-79's exact method (generalized eigenvalue problem on symmetrized lag-covariance
matrices), adapted for daily data (`tau=7`). Input = the continuous driver subset of `BASE_FX`,
**excluding** calendar cyclical encodings (`fx_DOY_sin/cos`) and binary flags (`fx_is_arable`,
`fx_is_growing`, `fx_is_winter`, `fx_grazing_active`) -- same exclusion logic D-79 used (these are
deterministic re-encodings/step functions, not physical measurements with real slow dynamics to
find). Fit **leak-free, per anchor, on pre-anchor history only**.

In [4]:
TICA_EXCLUDE = ["fx_DOY_sin", "fx_DOY_cos", "fx_is_arable", "fx_is_growing", "fx_is_winter",
                 "fx_grazing_active"]
TICA_INPUT_COLS = [c for c in BASE_FX if c not in TICA_EXCLUDE]
print(f"TICA input: {len(TICA_INPUT_COLS)} continuous driver columns (excluded {len(TICA_EXCLUDE)})")

TAU = 7          # 1-week lag (daily data) -- D-79 used tau=24h (1 day) on hourly data
N_COMPONENTS = 3


def fit_tica(hist_df, tau=TAU, n_components=N_COMPONENTS):
    """hist_df: pre-anchor, temporally contiguous daily rows, TICA_INPUT_COLS only.
    Returns (mean, std, eigvecs) for leak-free projection of any window via apply_tica()."""
    X = hist_df.values.astype(float)
    mean = np.nanmean(X, axis=0)
    std = np.nanstd(X, axis=0)
    std[std == 0] = 1.0
    Xz = np.nan_to_num((X - mean) / std, nan=0.0)

    X0, X1 = Xz[:-tau], Xz[tau:]
    C0 = X0.T @ X0 + X1.T @ X1
    Ctau = X0.T @ X1 + X1.T @ X0
    n_pairs = 2 * len(X0)
    C0 /= n_pairs
    Ctau /= n_pairs
    # Ridge regularization: with 43 continuous drivers many are near-collinear (e.g. fx_SWC_lag7/
    # 14/21/28/roll7/14 are all near-linear transforms of one signal), which left C0 not positive
    # definite -- caught directly by the smoke test's LinAlgError before the full sweep, not
    # silently worked around. Standard TICA/PCA stabilization (matches deeptime/PyEMMA's own
    # shrinkage handling): small relative ridge on the diagonal.
    C0 += 1e-6 * np.trace(C0) / C0.shape[0] * np.eye(C0.shape[0])

    vals, vecs = gen_eigh(Ctau, C0)
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:, order]
    return mean, std, vecs[:, :n_components], vals[:n_components]


def apply_tica(df, mean, std, vecs):
    """df: any window (hist or future), TICA_INPUT_COLS only. Returns (n_rows, n_components) array."""
    X = df.values.astype(float)
    Xz = np.nan_to_num((X - mean) / std, nan=0.0)
    return Xz @ vecs


TICA_COLS = [f"fx_tica{i+1}" for i in range(N_COMPONENTS)]

# Sanity check: Tower 4, anchor 2021 -- fit on pre-anchor history, report eigenvalues/implied timescales
_hist_test = T[4].loc[:_anchor_test, TICA_INPUT_COLS]
_mean, _std, _vecs, _vals = fit_tica(_hist_test)
_implied_ts = -TAU / np.log(np.clip(np.abs(_vals), 1e-6, 0.999999))
print(f"\nTICA sanity check (T4, anchor 2021, {len(_hist_test)} pre-anchor rows):")
print(f"  Eigenvalues (top {N_COMPONENTS}): {np.round(_vals, 3)}")
print(f"  Implied timescales (days): {np.round(_implied_ts, 1)}")
_top_loadings = pd.Series(_vecs[:, 0], index=TICA_INPUT_COLS).abs().sort_values(ascending=False).head(5)
print(f"  Top 5 |loadings| on IC1:\n{_top_loadings.round(3).to_string()}")


TICA input: 43 continuous driver columns (excluded 6)

TICA sanity check (T4, anchor 2021, 1811 pre-anchor rows):
  Eigenvalues (top 3): [0.991 0.977 0.914]
  Implied timescales (days): [751.7 305.6  78.3]
  Top 5 |loadings| on IC1:
fx_TS_roll14    0.183
fx_TS_lag14     0.157
fx_TS_lag21     0.155
fx_TS_lag28     0.097
fx_TS_lag7      0.082


## 5. Config assembly

4 configs: the champion baseline, each new family added on top individually, and both combined.

In [5]:
def build_configs():
    return {
        "BASE+species": CHAMPION_FX,
        "BASE+species+static_ar": CHAMPION_FX + STATIC_AR_COLS,
        "BASE+species+tica": CHAMPION_FX + TICA_COLS,
        "BASE+species+static_ar+tica": CHAMPION_FX + STATIC_AR_COLS + TICA_COLS,
    }

CONFIGS = build_configs()
for name, cols in CONFIGS.items():
    print(f"  {name}: {len(cols)} columns")


  BASE+species: 52 columns
  BASE+species+static_ar: 62 columns
  BASE+species+tica: 55 columns
  BASE+species+static_ar+tica: 65 columns


## 6. Per-anchor feature-frame builder

Builds the full covariate frame (real `fx_` + static_ar + tica) for one (tower, anchor), covering
both the pre-anchor history window and the 365-day future window -- static_ar/tica computed once
from pre-anchor data, static_ar broadcast constant, tica projected onto both windows via the same
fitted transform.

In [6]:
def build_frame_for_anchor(tower, anchor, target_dates):
    dft = T[tower]
    hist = dft.loc[:anchor]

    # static AR: HISTORY gets genuinely time-varying rolling stats (real context, not a constant
    # column -- see historical_static_ar_frame()'s docstring for why); FUTURE gets the single
    # frozen-at-anchor scalar, which is the deliberate, discussed design tradeoff.
    hist_ar = historical_static_ar_frame(hist["y_observed"])
    static_feats = static_ar_features(hist["y_observed"], anchor)

    # TICA: fit on pre-anchor history only, apply to both windows
    hist_tica_input = hist[TICA_INPUT_COLS]
    mean, std, vecs, _ = fit_tica(hist_tica_input)
    hist_tica = apply_tica(hist_tica_input, mean, std, vecs)
    future_tica_input = dft.loc[target_dates, TICA_INPUT_COLS]
    future_tica = apply_tica(future_tica_input, mean, std, vecs)

    hist_extra = pd.DataFrame(hist_tica, index=hist.index, columns=TICA_COLS)
    for c in STATIC_AR_COLS:
        hist_extra[c] = hist_ar[c].values

    future_extra = pd.DataFrame(future_tica, index=target_dates, columns=TICA_COLS)
    for k, v in static_feats.items():
        future_extra[k] = v

    hist_full = pd.concat([hist, hist_extra], axis=1)
    future_full = pd.concat([dft.loc[target_dates], future_extra], axis=1)
    return hist_full, future_full


# Smoke test: one (tower, anchor), all 4 configs, before committing to the full sweep
_anchor = pd.Timestamp("2021-12-16")
_target_dates = pd.date_range(_anchor + pd.Timedelta(days=1), periods=N_DAYS, freq="D")
_t0 = time.time()
_hist_full, _future_full = build_frame_for_anchor(4, _anchor, _target_dates)
for _cfg_name, _cols in CONFIGS.items():
    _chain = rr.tabicl_forecast(_hist_full["y_observed"], _hist_full[_cols], _future_full[_cols])
    print(f"  [smoke] T4 anchor2021 {_cfg_name}: chain len={len(_chain)}, "
          f"mean={_chain.mean():.2f}, {time.time()-_t0:.1f}s elapsed")
print("Smoke test OK")


GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]

  [smoke] T4 anchor2021 BASE+species: chain len=365, mean=29.70, 3.8s elapsed


GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

...

  [smoke] T4 anchor2021 BASE+species+tica: chain len=365, mean=32.84, 5.2s elapsed


GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.48it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.48it/s]

  [smoke] T4 anchor2021 BASE+species+static_ar+tica: chain len=365, mean=48.02, 5.9s elapsed
Smoke test OK


## 7. Full sweep: 3 towers x 5 anchors x 4 configs, TabICLv2 only

60 calls total. Saves raw chains (for later CQR/UQ work if this becomes a new champion) and
per-bin metrics (climatology-scored MASE primary, D-80 convention).

In [7]:
all_chains = []
all_metrics = []
t0 = time.time()
n_done = 0
n_total = len(TOWERS) * len(ANCHOR_YEARS) * len(CONFIGS)

for tower in TOWERS:
    dft = T[tower]
    for yr in ANCHOR_YEARS:
        anchor = pd.Timestamp(f"{yr}-12-16")
        target_dates = pd.date_range(anchor + pd.Timedelta(days=1), periods=N_DAYS, freq="D")

        hist_full, future_full = build_frame_for_anchor(tower, anchor, target_dates)
        y_true = dft["y_observed"].reindex(target_dates).values
        climatology = rr.doy_climatology(hist_full["y_observed"].dropna(), target_dates)
        anchor_val = dft.loc[anchor, "y_gapfilled"]
        persist = rr.chain_persistence(anchor_val, N_DAYS)

        for cfg_name, cols in CONFIGS.items():
            try:
                chain = rr.tabicl_forecast(hist_full["y_observed"], hist_full[cols], future_full[cols])
                yp = chain.reindex(target_dates).values

                cdf = chain.to_frame("pred").reset_index().rename(columns={"index": "date"})
                cdf["tower"] = tower; cdf["anchor_year"] = yr; cdf["config"] = cfg_name
                all_chains.append(cdf)

                bm = rr.bin_metrics(y_true, yp, target_dates, anchor, y_persist=climatology)
                bm["MASE_persistence"] = rr.bin_metrics(y_true, yp, target_dates, anchor, y_persist=persist)["MASE"]
                bm["tower"] = tower; bm["anchor_year"] = yr; bm["config"] = cfg_name
                all_metrics.append(bm)
            except Exception as e:
                print(f"    T{tower} {yr} {cfg_name} SKIPPED: {str(e)[:150]}")
            n_done += 1

        print(f"  T{tower} anchor {yr}: {len(CONFIGS)} configs done "
              f"({n_done}/{n_total}, {time.time()-t0:.0f}s elapsed)")

chains_df = pd.concat(all_chains, ignore_index=True)
metrics_df = pd.concat(all_metrics, ignore_index=True)
chains_df.to_csv(f"{RESULTS}/b16_tica_static_ar_chains.csv", index=False)
metrics_df.to_csv(f"{RESULTS}/b16_tica_static_ar_summary.csv", index=False)
print(f"\n[OK] Saved b16_tica_static_ar_chains/summary.csv, total {time.time()-t0:.0f}s")


GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]

GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]

...

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]

GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

  T9 anchor 2022: 4 configs done (60/60, 36s elapsed)

[OK] Saved b16_tica_static_ar_chains/summary.csv, total 36s


## 8. Results: n-weighted aggregate per config, climatology-scored MASE primary

In [8]:
def wavg(g, col):
    vals = g[col]
    if vals.isna().all():
        return np.nan
    w = g["n"]
    return (vals * w).sum() / w.sum() if w.sum() > 0 else np.nan

agg = metrics_df.groupby("config").apply(
    lambda g: pd.Series({
        "MASE_climatology": wavg(g, "MASE"),
        "MASE_persistence": wavg(g, "MASE_persistence"),
        "R2": wavg(g, "R2"),
        "RMSE": wavg(g, "RMSE"),
        "n_bins_valid": g["MASE"].notna().sum(),
    }), include_groups=False
).reset_index().sort_values("MASE_climatology")
print("Overall (all 3 towers, all 5 anchors, n-weighted mean across bins):")
print(agg.round(4).to_string(index=False))

print("\nPer-tower breakdown:")
agg_tower = metrics_df.groupby(["config", "tower"]).apply(
    lambda g: pd.Series({"MASE_climatology": wavg(g, "MASE"), "R2": wavg(g, "R2")}),
    include_groups=False
).reset_index()
print(agg_tower.round(4).to_string(index=False))


Overall (all 3 towers, all 5 anchors, n-weighted mean across bins):
                     config  MASE_climatology  MASE_persistence      R2    RMSE  n_bins_valid
          BASE+species+tica            0.7348            0.8560 -0.0937 52.9922          43.0
               BASE+species            0.7353            0.8556 -0.1070 53.3382          43.0
     BASE+species+static_ar            0.7358            0.8560 -0.0924 52.7976          43.0
BASE+species+static_ar+tica            0.7603            0.8871 -0.1559 53.4333          43.0

Per-tower breakdown:
                     config  tower  MASE_climatology      R2
               BASE+species      2            0.4133 -0.2105
               BASE+species      4            0.7926 -0.1023
               BASE+species      9            0.6747 -0.1009
     BASE+species+static_ar      2            0.4199 -0.2705
     BASE+species+static_ar      4            0.7901 -0.0552
     BASE+species+static_ar      9            0.6798 -0.1364
BASE+species+

## 9. Verdict

**Neither new family beats the standing champion, and combining them is actively worse -- a clean,
consistent negative result, not a mixed/ambiguous one.**

| Config | MASE (climatology) | R2 | Delta vs baseline |
|---|---|---|---|
| BASE+species (baseline) | 0.7353 | -0.1070 | -- |
| BASE+species+tica | 0.7348 | -0.0937 | -0.0005 (noise) |
| BASE+species+static_ar | 0.7358 | -0.0924 | +0.0005 (noise) |
| BASE+species+static_ar+tica | 0.7603 | -0.1559 | **+0.0250 (clearly worse)** |

**TICA alone**: essentially a wash (T2 0.4071 vs 0.4133, T4 0.7907 vs 0.7926, T9 0.6775 vs 0.6747)
-- replicates D-79's gap-filling finding (TICA-as-feature: wash) in a genuinely different task
(extrapolation, not interpolation). The prior did not transfer to a *positive* result here, but it
did transfer directionally (still a wash, not a reversal).

**Static AR features alone**: also essentially a wash (T2 slightly worse, T4/T9 flat), despite
being a real, if weak, per-anchor summary signal (fx_ch4_pre_mean7 etc. do vary across anchors --
confirmed via the sanity check in section 3). Consistent with the stated expectation going in:
TabICLv2 already sees the raw y_observed history natively, so explicit derived summary statistics
add little on top of what the model already has access to -- and the static (frozen-at-anchor)
design's known staleness-at-long-lead-times limitation (discussed with the user before building)
likely caps whatever value they could add even at short lead times.

**Combined (+static_ar+tica)**: clearly worse than either alone or the baseline, consistent across
all 3 towers (T2: 0.4300 vs 0.4133 baseline; T4: 0.8272 vs 0.7926; T9: 0.6830 vs 0.6747 -- not
driven by one tower). This matches a pattern already established elsewhere in this project's own
forecasting work: stacking many feature families together tends to hurt more than any individual
family helps (the original F-10/D-67 Stage-1 point-forecast check found BASE+ALL was the worst
single config by a wide margin) -- 13 extra columns (65 vs 52) appears to dilute/distract the
model's attention rather than add usable signal, even when neither family is harmful in isolation.

**Outcome: BASE+species remains the standing champion.** No new champion emerged from this search.
Both new feature families are negative results worth keeping on record (this is exactly the kind
of "checked directly, reported honestly" finding this project values) -- future feature-engineering
attempts on Track B should note that both an explicit-summary-statistics approach and a dynamics-
embedding approach have now been tried and found not to help, on top of an architecture that
already ingests raw target history natively.
